# 🎙️ Azərbaycan dili (AZJ) üçün VITS/MMS Səs Modelinin Fine-Tuning Edilməsi

Bu notebook `facebook/mms-tts-azj-script_latin` modelini (Meta-nın çoxdilli MMS TTS modelinin Azərbaycan versiyası) sizin öz səs dataset-iniz üzərində **fine-tune** etmək (yəni əlavə öyrətmək) üçün hazırlanıb.

## Bu notebook nə edir?

1. Sizin hazırladığınız audio + mətn dataset-ini yoxlayır və doğru formata salır.
2. Meta-nın orijinal MMS modelini yükləyir.
3. Fine-tuning üçün lazım olan `finetune-hf-vits` kitabxanasını qurur və uyğunlaşdırır.
4. Modeli sizin dataset-iniz üzərində öyrədir (təlim).
5. Nəticəni (yeni səsi) test edir və istəsəniz Hugging Face Hub-a yükləyir.

## Əvvəlcədən lazım olanlar

- **Google Colab** hesabı (GPU aktiv edilmiş — `Runtime → Change runtime type → GPU`).
- `hf_dataset.zip` adlı fayl — bu, Hugging Face `datasets` formatında saxlanmış audio+mətn dataset-idir. Bu faylı Colab-ın sol tərəfindəki fayl panelindən `/content/` qovluğuna yükləyin (sürüşdürüb ata bilərsiniz).
- Dataset-də ən azı bir neçə yüz qısa audio nümunəsi (adətən 1-12 saniyə arası) və onlara uyğun mətn olmalıdır.

## Bu versiyada nə düzəldilib?

Orijinal, sınaq-xəta ilə yazılmış notebook-dan fərqli olaraq, bu versiya:

- Bütün təkrarlanan/artıq hüceyrələr təmizlənib, hər addım yalnız **bir dəfə**, aydın izahla verilib.
- `transformers` / `accelerate` versiyaları `finetune-hf-vits` reposunun kodu ilə **uyğun** versiyalara sabitlənib (yeni versiyalar `TrainingArguments`-dən `overwrite_output_dir` sahəsini silib, bu da xətaya səbəb olurdu).
- Dataset düzgün `train`/`test` bölgüsünə ayrılır və `load_dataset`-in taniya biləcəyi formatda diskə yazılır (`eval` yox, `test` adı istifadə olunur).
- Təlimin kəsilməsi/davam etdirilməsi üçün təkrar istifadə edilə bilən, təmiz bir hissə əlavə olunub.
- Konfiqurasiyadakı **hər bir** parametr üçün sadə dildə izah verilib (aşağıda, 13-cü bölmədə).

Ətraflı izah və fayl strukturu üçün notebook-la birgə gələn **README.md** faylına baxın.

## 1. GPU yoxlanması

Bu hüceyrə Colab-a ayrılan GPU-nun mövcud olduğunu göstərir. Fine-tuning GPU olmadan praktiki olaraq mümkün deyil (CPU ilə həftələr çəkər). Əgər `NVIDIA-SMI` xəta versə, `Runtime → Change runtime type → GPU` seçin.

In [ ]:
!nvidia-smi

## 2. Lazım olan əsas paketləri quraşdır

Burada dataset və audio emalı üçün lazım olan əsas Python kitabxanaları quraşdırılır. `transformers`/`accelerate` kimi təlimlə birbaşa bağlı paketləri isə **7-ci bölmədə**, xüsusi (sabit) versiyalarla quraşdıracağıq — buna görə burada onları qurmuruq.

In [ ]:
!pip install -q datasets soundfile librosa pandas matplotlib

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA mövcuddur:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU YOXDUR! Runtime -> Change runtime type -> GPU seçin.")

In [ ]:
import os

print(os.listdir("/content"))
# "hf_dataset.zip" bu siyahıda görünməlidir.
# Görünmürsə, onu Colab-ın sol fayl panelindən /content/ qovluğuna yükləyin.

## 3. Dataset ZIP faylını aç

`hf_dataset.zip` Hugging Face `datasets` kitabxanasının formatında saxlanmış audio+mətn cütlüklərini ehtiva edir. Aşağıdakı hüceyrə onu `/content/dataset` qovluğuna açır və içindəki fayl strukturunu göstərir (ora `dataset_info.json`, `*.arrow` və `state.json` kimi fayllar daxildir olmalıdır).

In [ ]:
import zipfile
import os

zip_path = "/content/hf_dataset.zip"
extract_dir = "/content/dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

## 4. Dataset-i aç və keyfiyyətini yoxla

Bu bölmədə dataset-in düzgün açıldığını, nə qədər nümunə olduğunu, nümunələrin necə göründüyünü (mətn + audio) və audio uzunluqlarının statistikasını yoxlayırıq. Bu, təlimə başlamazdan əvvəl datanın "sağlam" olduğuna əmin olmaq üçün vacibdir.

**Qeyd:** aşağıdakı `dataset_path` sizin zip faylınızın daxili qovluq adına uyğun olmalıdır (yuxarıdakı 3-cü addımın çıxışında görünən adı istifadə edin, məsələn `"hf_dataset - Copy"`).

In [ ]:
from datasets import load_from_disk

dataset_path = "/content/dataset/hf_dataset - Copy"  # lazım olsa yolu dəyişin

dataset = load_from_disk(dataset_path)

print(dataset)
print()
print("Sütunlar:", dataset.column_names)
print("Xüsusiyyətlər (features):", dataset.features)
print("Dataset ölçüsü:", len(dataset))

In [ ]:
sample = dataset[0]

print(sample.keys())
print()
print("TEXT:", sample["text"])
print()
print("AUDIO metadata:", sample["audio"])

In [ ]:
from IPython.display import Audio, display

sample = dataset[0]

display(Audio(
    sample["audio"]["array"],
    rate=sample["audio"]["sampling_rate"]
))
print(sample["text"])

In [ ]:
import numpy as np

durations = []
for item in dataset:
    audio = item["audio"]["array"]
    sr = item["audio"]["sampling_rate"]
    durations.append(len(audio) / sr)

durations = np.array(durations)

print("Nümunə sayı:", len(durations))
print("Minimum uzunluq:", round(durations.min(), 2), "saniyə")
print("Maksimum uzunluq:", round(durations.max(), 2), "saniyə")
print("Orta uzunluq:", round(durations.mean(), 2), "saniyə")

# Qeyd: aşağıda (13-cü bölmədə) konfiqurasiyada "min_duration_in_seconds" və
# "max_duration_in_seconds" dəyərlərini bu statistikaya uyğun seçmək tövsiyə olunur.

## 5. Dataset-i `train` / `test` bölmələrinə ayır

**Niyə lazımdır?** Fine-tuning skripti (`run_vits_finetuning.py`) dataset-i `load_from_disk` ilə yox, `datasets.load_dataset(...)` funksiyası ilə açır. Bu funksiya yalnız `train`, `test` və `validation` adlı bölmələri (split) avtomatik tanıyır — `eval` adını **tanımır**. Ona görə də dataset-i qabaqcadan `train`/`test`-ə bölüb, düzgün formatda diskə yazmalıyıq.

Aşağıda dataset 90% `train` / 10% `test` nisbətində bölünür və `/content/azj_tts_dataset` qovluğuna, `load_dataset`-in düzgün oxuya biləcəyi formatda yazılır.

In [ ]:
from datasets import load_from_disk
import shutil
import os

# 1) Mövcud (bölünməmiş) dataset-i aç
dataset_for_split = load_from_disk(dataset_path)

# 2) train / test bölgüsü yarat (90% / 10%)
split_dataset = dataset_for_split.train_test_split(test_size=0.1, seed=42)
print(split_dataset)   # DatasetDict({'train': ..., 'test': ...})

# 3) Köhnə (yarımçıq/uyğunsuz) qovluğu təmizlə
shutil.rmtree("/content/azj_tts_dataset", ignore_errors=True)
os.makedirs("/content/azj_tts_dataset", exist_ok=True)

# 4) Hər bölməni load_dataset-in tanıya biləcəyi struktur ilə saxla
split_dataset.save_to_disk("/content/azj_tts_dataset")

print("✅ Dataset '/content/azj_tts_dataset' qovluğuna train/test olaraq yazıldı.")

In [ ]:
# Yoxlama: dataset load_dataset() ilə düzgün oxunurmu?
from datasets import load_dataset

check_ds = load_dataset("/content/azj_tts_dataset")
print(check_ds)
print()
print("Nümunə mətn:", check_ds["train"][0]["text"])
print("Nümunə sampling_rate:", check_ds["train"][0]["audio"]["sampling_rate"])

## 6. Meta-nın əsas MMS modelini yüklə

Fine-tuning üçün Meta-nın orijinal, tam (generator **və** discriminator daxil olmaqla) checkpoint faylı lazımdır — çünki VITS modeli "adversarial" (GAN-vari) üsulla öyrədilir və discriminator hissəsi olmadan keyfiyyətli fine-tuning mümkün deyil. Bu fayl Hugging Face-dəki modeldə **yoxdur**, ona görə onu ayrıca Meta-nın öz serverindən yükləyirik.

In [ ]:
!wget -O /content/azj-script_latin-full.tar.gz \
    https://dl.fbaipublicfiles.com/mms/tts/full_model/azj-script_latin.tar.gz

In [ ]:
!mkdir -p /content/mms-azj-full
!tar -xzf /content/azj-script_latin-full.tar.gz -C /content/mms-azj-full
!find /content/mms-azj-full -maxdepth 3 -type f

## 7. Fine-tuning reposunu klonla və mühiti hazırla

`finetune-hf-vits` — Hugging Face `transformers` kitabxanasındakı VITS modelini fine-tune etmək üçün yazılmış açıq mənbəli skript dəstidir. Onu GitHub-dan klonlayıb, tələb olunan paketləri quraşdırırıq.

In [ ]:
!git clone https://github.com/ylacombe/finetune-hf-vits.git /content/finetune-hf-vits
%cd /content/finetune-hf-vits
!pip install -q -r requirements.txt

### ⚠️ Vacib düzəliş: `transformers` / `accelerate` versiyalarını sabitlə

`requirements.txt` faylında `transformers` üçün konkret versiya göstərilmədiyi üçün yuxarıdakı `pip install` ən son `transformers` versiyasını (5.x) quraşdırır. Amma bu repo 2023-cü ilin `transformers` API-sinə uyğun yazılıb və yeni versiyalarda `TrainingArguments` sinfindən `overwrite_output_dir` sahəsi silinib. Nəticədə təlimi başladanda

```
ValueError: Some keys are not used by the HfArgumentParser
```

xətası alınır. Bunu **köhnə, repo ilə uyğun** versiyalara keçərək həll edirik. Bu, notebook-un ən vacib "düzəliş" hüceyrələrindən biridir — silsəniz və ya versiyanı dəyişsəniz, təlim işə düşməyə bilər.

In [ ]:
!pip install -q "transformers==4.41.2" "accelerate==0.29.3" "huggingface_hub==0.24.6"
print("✅ transformers, accelerate və huggingface_hub uyğun versiyalara sabitləndi.")

## 8. `monotonic_align` modulunu dərlə (compile et)

VITS modelinin "monotonic alignment search" adlı Cython/C++ kodu var — bu, mətn və audio arasındakı uyğunlaşmanı tapan alqoritmdir. İstifadədən əvvəl yerli olaraq dərlənməlidir (bir dəfə edilir, sonra fayl sistemdə qalır).

In [ ]:
%cd /content/finetune-hf-vits/monotonic_align

!mkdir -p monotonic_align
!python setup.py build_ext --inplace
!find . -maxdepth 2 -type f

## 9. Hugging Face-dən uyğun generator checkpoint-i yüklə

`huggingface_hub` artıq 7-ci bölmədə `transformers`/`accelerate` ilə birgə sabit versiyaya (`0.24.6`) sabitlənib. Burada onu **təkrar yeniləmirik** — əks halda `transformers==4.41.2`-nin tələb etdiyi versiya aralığı (`huggingface-hub>=0.23.0,<1.0`) pozula bilər.

Bu addımda `facebook/mms-tts-azj-script_latin` modelinin Hugging Face formatındakı generator hissəsini yükləyirik (config, tokenizer və çəkilər).

In [ ]:
import huggingface_hub
print("huggingface_hub versiyası:", huggingface_hub.__version__)

In [ ]:
from huggingface_hub import snapshot_download

HF_MODEL = "facebook/mms-tts-azj-script_latin"

generator_path = snapshot_download(
    repo_id=HF_MODEL,
    local_dir="/content/mms-azj-hf"
)

print("Model yükləndi:", generator_path)
!find /content/mms-azj-hf -maxdepth 2 -type f | sort

## 10. Discriminator checkpoint-i Hugging Face formatına çevir

6-cı addımda yüklədiyimiz Meta checkpoint-indəki discriminator çəkilərini (`D_100000.pth`) `finetune-hf-vits` reposunun gözlədiyi Hugging Face formatına çeviririk. Nəticə `/content/azj-mms-training` qovluğuna yazılır — bu, artıq **fine-tuning üçün tam hazır olan başlanğıc model** olacaq (həm generator, həm də discriminator daxil).

In [ ]:
%cd /content/finetune-hf-vits

!python convert_original_discriminator_checkpoint.py \
    --checkpoint_path /content/mms-azj-full/azj-script_latin/D_100000.pth \
    --generator_checkpoint_path /content/mms-azj-hf \
    --pytorch_dump_folder_path /content/azj-mms-training

## 11. Tokenizer `pad_token_id` düzəlişi

Bəzi MMS checkpoint-lərində model konfiqurasiyasındakı `pad_token_id` sahəsi tokenizer-in həqiqi pad token id-si ilə üst-üstə düşmür. Bu uyğunsuzluq təlim zamanı səhv nəticələrə səbəb ola bilər, ona görə əvvəlcə yoxlayır, sonra düzəldirik (`0`-a təyin edirik, çünki `finetune-hf-vits` bunu gözləyir).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/content/mms-azj-hf")

print("pad_token:", tokenizer.pad_token, "| pad_token_id:", tokenizer.pad_token_id)
print("unk_token:", tokenizer.unk_token, "| unk_token_id:", tokenizer.unk_token_id)

In [ ]:
import json

config_path = "/content/azj-mms-training/config.json"

with open(config_path, "r", encoding="utf-8") as f:
    model_config = json.load(f)

model_config["pad_token_id"] = 0

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=2, ensure_ascii=False)

print("✅ pad_token_id =", model_config["pad_token_id"])
print("vocab_size:", model_config.get("vocab_size"))

## 12. Modelin (sabitlənmiş `transformers` versiyası ilə) düzgün yükləndiyini yoxla

Təlimə başlamazdan əvvəl modelin `VitsModelForPreTraining` sinfi ilə heç bir xəta vermədən yükləndiyinə əmin oluruq.

In [ ]:
%cd /content/finetune-hf-vits

from utils.modeling_vits_training import VitsModelForPreTraining

model = VitsModelForPreTraining.from_pretrained("/content/azj-mms-training")

print("✅ MODEL UĞURLA YÜKLƏNDİ")
print("Parametr sayı:", sum(p.numel() for p in model.parameters()))

## 13. Fine-tuning konfiqurasiyasını (`azj_finetune.json`) yarat

Bu, notebook-un **ən vacib** hüceyrəsidir — bütün təlim parametrləri burada təyin olunur. Hər parametrin nə etdiyinin **tam izahı README.md faylındadır** ("Konfiqurasiya parametrləri" bölməsi). Qısaca:

- `dataset_name` → 5-ci addımda yaratdığımız `/content/azj_tts_dataset` yoluna işarə edir.
- `eval_split_name` → `"test"` olaraq təyin olunub (`"eval"` yox), çünki `train_test_split()` defolt olaraq `test` adında bölmə yaradır və `load_dataset` yalnız bu adı tanıyır.
- `num_train_epochs`, `learning_rate`, `batch_size` kimi parametrləri öz dataset ölçünüzə və GPU yaddaşınıza görə dəyişə bilərsiniz.

Aşağıdakı dəyərləri dəyişməzdən əvvəl mütləq README.md-ə baxın — bəzi parametrlər (məsələn `eval_split_name`, `model_name_or_path`) səhv dəyişdirilsə, təlim ümumiyyətlə başlamır.

In [ ]:
import json

config = {
    "project_name": "azj_single_speaker_mms",

    "model_name_or_path": "/content/azj-mms-training",
    "dataset_name": "/content/azj_tts_dataset",

    "audio_column_name": "audio",
    "text_column_name": "text",

    "train_split_name": "train",
    "eval_split_name": "test",

    "output_dir": "/content/azj_vits_finetuned",
    "overwrite_output_dir": True,

    "do_train": True,
    "do_eval": True,

    "num_train_epochs": 50,

    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 1,

    "learning_rate": 2e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_ratio": 0.01,

    "preprocessing_num_workers": 2,
    "dataloader_num_workers": 2,

    "max_duration_in_seconds": 12,
    "min_duration_in_seconds": 1.5,
    "max_tokens_length": 450,

    "full_generation_sample_text":
        "Dünyanın sonu nə deməkdir ki, dünyanın sonu o demək deyil ki, "
        "su gəlib dünyanı bassın, evlər və insanlar sualtında qalsın, yox.",

    "do_step_schedule_per_epoch": True,
    "lr_decay": 0.999875,

    "weight_disc": 3,
    "weight_fmaps": 1,
    "weight_gen": 1,
    "weight_kl": 1.5,
    "weight_duration": 1,
    "weight_mel": 35,

    "fp16": True,

    "logging_steps": 10,
    "eval_steps": 25,
    "save_steps": 25,
    "save_total_limit": 5,

    "report_to": [],
    "seed": 42,

    "preprocessing_only": False
}

config_path = "/content/finetune-hf-vits/azj_finetune.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✅ CONFIG HAZIRDIR:", config_path)

In [ ]:
import json

with open("/content/finetune-hf-vits/azj_finetune.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

print("model:", cfg["model_name_or_path"])
print("dataset:", cfg["dataset_name"])
print("train split:", cfg["train_split_name"])
print("eval split:", cfg["eval_split_name"])
print("epochs:", cfg["num_train_epochs"])
print("batch size:", cfg["per_device_train_batch_size"])
print("learning rate:", cfg["learning_rate"])
print("save_total_limit:", cfg.get("save_total_limit"))

## 14. Skriptə kiçik uyğunlaşdırma düzəlişləri (patch-lər)

`finetune-hf-vits` reposu bir neçə il əvvəl yazıldığı üçün bugünkü mühitlə tam uyğun işləməyən 3 kiçik yer var. Aşağıda bunları avtomatik düzəldirik:

1. **Telemetry çağırışının silinməsi** — `transformers` skriptin hər işə düşməsində statistika göndərməyə çalışır; bunu söndürürük (xarici şəbəkə çağırışının qarşısını almaq üçün, məcburi deyil).
2. **`speaker_id` sahəsinin düzəlişi** — tək spikerli dataset-lərdə (bizim halımızda olduğu kimi) `speaker_id` sahəsi olmadıqda skript xəta verirdi; bunu təhlükəsiz hala gətiririk.
3. **`plot.py` faylının tam təmiz versiya ilə əvəzlənməsi** — alignment/spektroqram qrafiklərini çəkən köməkçi fayldakı indentasiya və format problemlərini aradan qaldırırıq.

In [ ]:
from pathlib import Path

path = Path("/content/finetune-hf-vits/run_vits_finetuning.py")
text = path.read_text(encoding="utf-8")

# 1) Telemetry importunu və çağırışını sil
text = text.replace(
    "from transformers.utils import send_example_telemetry\n", ""
)
text = text.replace(
    '    # Sending telemetry. Tracking the example usage helps us better allocate resources to maintain them. The\n'
    '    # information sent is the one passed as arguments along with your Python/PyTorch versions.\n'
    '    send_example_telemetry("run_vits_finetuning", model_args, data_args)\n',
    ""
)

# 2) speaker_id sahəsini təhlükəsiz hala gətir
old_block = (
    '        batch["speaker_id"] = (\n'
    '            torch.tensor([feature["speaker_id"] for feature in features]) if "speaker_id" in features[0] else None\n'
    '        )\n'
)
new_block = (
    '        if "speaker_id" in features[0]:\n'
    '            batch["speaker_id"] = torch.tensor([feature["speaker_id"] for feature in features])\n'
)
text = text.replace(old_block, new_block)
text = text.replace('speaker_id=batch["speaker_id"],', 'speaker_id=batch.get("speaker_id"),')

path.write_text(text, encoding="utf-8")
print("✅ Telemetry və speaker_id patch-ləri tətbiq olundu.")

In [ ]:
plot_file = "/content/finetune-hf-vits/utils/plot.py"

with open(plot_file, "w", encoding="utf-8") as f:
    f.write('''import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

def plot_alignment_to_numpy(alignment, info=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(alignment, aspect=\'auto\', origin=\'lower\', interpolation=\'none\')
    fig.colorbar(im, ax=ax)
    xlabel = \'Decoder timestep\'
    if info is not None:
        xlabel += \'\\n\\n\' + info
    ax.set_xlabel(xlabel)
    ax.set_ylabel(\'Encoder timestep\')
    ax.yaxis.set_ticks_position(\'left\')
    ax.xaxis.set_ticks_position(\'bottom\')
    fig.canvas.draw()
    data = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    w, h = fig.canvas.get_width_height()
    data = data.reshape((h, w, 4))
    data = data[:, :, :3]
    data = np.ascontiguousarray(data)
    plt.close(fig)
    return data

def plot_spectrogram_to_numpy(spectrogram):
    fig, ax = plt.subplots(figsize=(12, 3))
    im = ax.imshow(spectrogram, aspect="auto", origin="lower", interpolation=\'none\')
    plt.colorbar(im, ax=ax)
    plt.xlabel("Frames")
    plt.ylabel("Channels")
    plt.tight_layout()
    fig.canvas.draw()
    data = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    w, h = fig.canvas.get_width_height()
    data = data.reshape((h, w, 4))
    data = data[:, :, :3]
    data = np.ascontiguousarray(data)
    plt.close(fig)
    return data
''')

print("✅ plot.py təmiz versiya ilə əvəz olundu.")

## 15. Skriptin idxal (import) olunduğunu yoxla

Əsl təlimi başlatmazdan əvvəl skriptin heç bir sintaksis/idxal xətası olmadan yükləndiyinə əmin oluruq.

In [ ]:
%cd /content/finetune-hf-vits
!python -c "import run_vits_finetuning; print('✅ SCRIPT IMPORT OK')"

## 16. Fine-tuning-i başlat

Bu, əsas təlim hüceyrəsidir. `accelerate launch` GPU-nu düzgün konfiqurasiya ilə istifadə edərək `run_vits_finetuning.py` skriptini, 13-cü addımda yaratdığımız `azj_finetune.json` konfiqurasiyası ilə işə salır.

**Diqqət:** dataset ölçüsündən və `num_train_epochs` dəyərindən asılı olaraq bu, saatlarla çəkə bilər. Colab-ın pulsuz versiyasında sessiya arası-arası kəsilə bilər — əgər bu baş verərsə, aşağıdakı **17-ci bölmədən** istifadə edərək təlimi olduğu yerdən davam etdirə bilərsiniz.

In [ ]:
%cd /content/finetune-hf-vits

!accelerate launch \
    --num_processes 1 \
    --num_machines 1 \
    --mixed_precision fp16 \
    run_vits_finetuning.py azj_finetune.json

## 17. Təlim kəsilibsə: son checkpoint-dən davam etdirmək

Colab sessiyası kəsildikdə (internet, timeout, GPU limiti və s.), təlimi sıfırdan başlamaq lazım deyil. `output_dir` (`/content/azj_vits_finetuned`) qovluğunda `checkpoint-<addım_sayı>` adlı yarımçıq təlim vəziyyətləri saxlanılır (`save_steps` parametrinə uyğun olaraq, hər 25 addımdan bir).

Aşağıdakı hüceyrə **avtomatik olaraq ən son checkpoint-i tapır**, onu konfiqurasiyaya (`resume_from_checkpoint`) yazır və təlimi yenidən başladır. Bu hüceyrəni istənilən qədər dəfə, təlim kəsildikcə təkrar işə sala bilərsiniz.

In [ ]:
import json
import glob

checkpoints = sorted(
    glob.glob("/content/azj_vits_finetuned/checkpoint-*"),
    key=lambda x: int(x.split("-")[-1])
)

config_path = "/content/finetune-hf-vits/azj_finetune.json"

with open(config_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

if checkpoints:
    latest = checkpoints[-1]
    print("Ən son tapılan checkpoint:", latest)
    cfg["resume_from_checkpoint"] = latest
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    print("✅ Konfiqurasiya yeniləndi — təlim bu checkpoint-dən davam edəcək.")
else:
    cfg.pop("resume_from_checkpoint", None)
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    print("ℹ️ Heç bir checkpoint tapılmadı — təlim sıfırdan başlayacaq.")

In [ ]:
%cd /content/finetune-hf-vits

!accelerate launch \
    --num_processes 1 \
    --num_machines 1 \
    --mixed_precision fp16 \
    run_vits_finetuning.py azj_finetune.json

**Sıfırdan yenidən başlamaq istəsəniz** (məsələn, konfiqurasiyanı köklü şəkildə dəyişdinizsə), aşağıdakı hüceyrəni açıb işə sala bilərsiniz. Bu, `output_dir`-i tamamilə siləcək. **Diqqətlə istifadə edin — geri qaytarıla bilməz.**

```python
# import shutil
# shutil.rmtree("/content/azj_vits_finetuned", ignore_errors=True)
# print("Köhnə təlim qovluğu silindi, təlim sıfırdan başlaya bilər.")
```

## 18. Disk yerinin idarə edilməsi

Hər checkpoint bir neçə yüz MB–1 GB yer tuta bilər. Colab-ın disk yeri məhduddur, ona görə:

- Konfiqurasiyada `save_total_limit: 5` təyin etmişik (13-cü addımda) — bu, `finetune-hf-vits`-ə **avtomatik olaraq yalnız son 5 checkpoint-i saxlamağı** və köhnələri silməyi tapşırır.
- Aşağıdakı hüceyrələr isə əl ilə yoxlama və təmizlik üçündür: mövcud checkpoint-lərin ölçüsünü göstərmək, artıq lazım olmayan böyük faylları (tar.gz, zip) silmək və checkpoint-lərin bütöv (zədəsiz) olduğunu yoxlamaq.

In [ ]:
!df -h /content

In [ ]:
# Artıq açılmış/istifadə olunmuş böyük fayllar (lazım deyil, yer tuturlar)
!rm -f /content/azj-script_latin-full.tar.gz
!rm -f /content/hf_dataset.zip

In [ ]:
import os
import glob

checkpoints = sorted(
    glob.glob("/content/azj_vits_finetuned/checkpoint-*"),
    key=lambda x: int(x.split("-")[-1])
)

print("Mövcud checkpoint-lər:")
for cp in checkpoints:
    # Qovluğun ümumi ölçüsü
    total = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fs in os.walk(cp) for f in fs
    )
    optimizer_ok = os.path.exists(os.path.join(cp, "optimizer.pt"))
    status = "✅ tam" if optimizer_ok else "⚠️ optimizer.pt yoxdur (zədəli ola bilər)"
    print(f"  {cp} → {total // (1024*1024)} MB → {status}")

## 19. Nəticələri test et

Təlim bitdikdən (və ya arzu etdiyiniz checkpoint-ə çatdıqdan) sonra modelin nə səsləndirdiyini yoxlaya bilərsiniz. Aşağıda 3 fərqli test variantı var:

1. Əsas (`output_dir`-dəki son) modeldən tək bir cümlə səsləndirmək.
2. Son bir neçə checkpoint-i **müqayisə** etmək (hansının daha yaxşı səsləndiyini seçmək üçün faydalıdır).
3. **Base model** (heç fine-tune edilməmiş orijinal MMS) ilə **fine-tuned model**-i birbaşa müqayisə etmək.

**Qeyd:** checkpoint qovluqlarında bəzən `config.json` olmur (yalnız root `output_dir`-də olur). Aşağıdakı kod bunu avtomatik aşkar edib lazım olduqda root-dakı `config.json`-u checkpoint-ə kopyalayır.

In [ ]:
import os
import shutil
import glob

# Son 5 checkpoint-ə root-dakı config.json-u kopyala (yoxdursa)
checkpoints = sorted(
    glob.glob("/content/azj_vits_finetuned/checkpoint-*"),
    key=lambda x: int(x.split("-")[-1])
)[-5:]

source_config = "/content/azj_vits_finetuned/config.json"

if os.path.exists(source_config):
    for cp in checkpoints:
        dest = os.path.join(cp, "config.json")
        if not os.path.exists(dest):
            shutil.copy(source_config, dest)
            print(f"✅ config.json kopyalandı: {cp}")
else:
    print("⚠️ Root config.json tapılmadı — model hələ təlim başa çatmayıb ola bilər.")

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch
import soundfile as sf
from IPython.display import Audio, display

# Fine-tuned modeli yüklə (son vəziyyət)
model_path = "/content/azj_vits_finetuned"
model = VitsModel.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Test mətni — istədiyinizi yaza bilərsiniz
text = "Salam, bu mənim incə tənzimlənmiş səs modelimin sınaq nümunəsidir."

inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

audio = outputs.waveform[0].squeeze().cpu().numpy()
sf.write("/content/final_test.wav", audio, samplerate=16000)
display(Audio("/content/final_test.wav"))
print("✅ Səs yaradıldı: /content/final_test.wav")

In [ ]:
import os
import glob
import torch
import soundfile as sf
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display

# Son 5 checkpoint-i tap və hər birindən eyni cümləni səsləndir (müqayisə üçün)
checkpoints = sorted(
    glob.glob("/content/azj_vits_finetuned/checkpoint-*"),
    key=lambda x: int(x.split("-")[-1])
)[-5:]

test_text = "Salam, bu mənim incə tənzimlənmiş səs modelimin sınaq nümunəsidir."
audio_files = []

for i, cp_path in enumerate(checkpoints):
    cp_num = cp_path.split("-")[-1]
    print(f"[{i+1}/{len(checkpoints)}] checkpoint-{cp_num} yüklənir...")
    try:
        model = VitsModel.from_pretrained(cp_path)
        tokenizer = AutoTokenizer.from_pretrained(cp_path)

        inputs = tokenizer(test_text, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)

        audio = outputs.waveform[0].squeeze().cpu().numpy()
        filename = f"/content/test_checkpoint_{cp_num}.wav"
        sf.write(filename, audio, samplerate=16000)
        audio_files.append((cp_num, filename))

        del model, tokenizer
        torch.cuda.empty_cache()
        print(f"  ✅ hazırdır: {filename}")
    except Exception as e:
        print(f"  ❌ checkpoint-{cp_num} yüklənə bilmədi: {e}")

print("\n📢 Bütün səsləri dinləyib müqayisə edin:")
for cp_num, filename in audio_files:
    print(f"\n🔊 checkpoint-{cp_num}:")
    display(Audio(filename))

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch
import soundfile as sf
from IPython.display import Audio, display

text = "Texnologiya inkişaf etdikcə, süni intellekt vasitəsilə yaradılan səslər getdikcə daha təbii olur."

# 1) Base model (heç fine-tune edilməməmiş orijinal MMS)
print("📢 Base model (orijinal MMS) yüklənir...")
base_model = VitsModel.from_pretrained("facebook/mms-tts-azj-script_latin")
base_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-azj-script_latin")

inputs = base_tokenizer(text, return_tensors="pt")
with torch.no_grad():
    base_outputs = base_model(**inputs)

base_audio = base_outputs.waveform[0].squeeze().cpu().numpy()
sf.write("/content/base_model_compare.wav", base_audio, samplerate=16000)
print("✅ Base model səsi hazırdır")
display(Audio("/content/base_model_compare.wav"))

del base_model, base_tokenizer
torch.cuda.empty_cache()

# 2) Fine-tuned model
print("\n📢 Fine-tuned model yüklənir...")
ft_model = VitsModel.from_pretrained("/content/azj_vits_finetuned")
ft_tokenizer = AutoTokenizer.from_pretrained("/content/azj_vits_finetuned")

inputs = ft_tokenizer(text, return_tensors="pt")
with torch.no_grad():
    ft_outputs = ft_model(**inputs)

ft_audio = ft_outputs.waveform[0].squeeze().cpu().numpy()
sf.write("/content/finetuned_model_compare.wav", ft_audio, samplerate=16000)
print("✅ Fine-tuned model səsi hazırdır")
display(Audio("/content/finetuned_model_compare.wav"))

print("\n🎯 İndi hər iki səsi dinləyib fərqi qiymətləndirin.")

## 20. Modeli Hugging Face Hub-a yüklə (istəyə bağlı)

Nəticədən razısınızsa, modeli Hugging Face Hub-da öz hesabınızda paylaşa (və ya özünüz üçün saxlaya) bilərsiniz. Bunun üçün Hugging Face hesabınız və **write** icazəli token lazımdır (https://huggingface.co/settings/tokens ünvanından yarada bilərsiniz).

Aşağıdakı hüceyrə token vasitəsilə daxil olmağı istəyəcək, sonra istifadəçi adınızı avtomatik token-dən götürüb, `<istifadəçi_adı>/azj-tts-finetuned` adlı repo yaradacaq (əgər `repo_name` dəyişənini dəyişməmisinizsə) və modeli oraya yükləyəcək.

In [ ]:
from huggingface_hub import HfApi, create_repo, notebook_login

# 1. Hugging Face-ə daxil ol (token istəyəcək)
notebook_login()

# 2. Token-dən istifadəçi adını al
api = HfApi()
your_username = api.whoami()["name"]
print(f"✅ İstifadəçi adı: {your_username}")

# 3. Repo adı — istəsəniz dəyişin
repo_name = "azj-tts-finetuned"
repo_id = f"{your_username}/{repo_name}"

# 4. Repo yarat (əgər artıq varsa, xəta vermir)
try:
    create_repo(repo_id, exist_ok=True, repo_type="model")
    print(f"✅ Repo hazırdır: {repo_id}")
except Exception as e:
    if "409" in str(e):
        print(f"⚠️ Repo artıq mövcuddur: {repo_id}")
    else:
        raise e

# 5. Qovluğu yüklə
api.upload_folder(
    folder_path="/content/azj_vits_finetuned",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Fine-tuned Azərbaycan dili TTS modeli (MMS/VITS)"
)

print(f"\n✅ Model UĞURLA yükləndi!")
print(f"🔗 Link: https://huggingface.co/{repo_id}")

## 21. Yüklənmiş modeli Hub-dan istifadə et

Model Hub-a yükləndikdən sonra, onu istənilən başqa layihədən (və ya bu notebook-un özündən, yeni sessiyada) birbaşa `repo_id` ilə yükləyib istifadə edə bilərsiniz. Aşağıda `repo_id` dəyişənini öz repo adınızla əvəz edin.

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch
import soundfile as sf
from IPython.display import Audio, display

repo_id = "istifadeci_adiniz/azj-tts-finetuned"  # öz repo id-nizlə əvəz edin

model = VitsModel.from_pretrained(repo_id)
tokenizer = AutoTokenizer.from_pretrained(repo_id)

text = "Salam, bu mənim Hugging Face-ə yüklədiyim incə tənzimlənmiş səs modelimdir."

inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

audio = outputs.waveform[0].squeeze().cpu().numpy()
sf.write("/content/hub_model_test.wav", audio, samplerate=16000)
display(Audio("/content/hub_model_test.wav"))
print("✅ Hub-dakı model uğurla test edildi.")

## 22. Yekun və problemlərin həlli

**Ən çox rastlanan problemlər:**

| Problem | Səbəb | Həll |
|---|---|---|
| `ValueError: Some keys are not used by the HfArgumentParser` | `transformers` versiyası çox yenidir | 7-ci addımdakı versiya sabitləmə hüceyrəsini yenidən işə salın |
| `load_dataset` boş/xəta verir | split adı `eval` olub, `test` olmayıb | 5-ci addımı təkrar işə salıb `eval_split_name: "test"` təyin edin |
| Təlim Colab kəsilməsi səbəbiylə dayanıb | Colab sessiyası vaxtı bitib | 17-ci addımdakı "resume" hüceyrələrini işə salın |
| Disk yeri bitir | Çoxlu checkpoint saxlanılır | 18-ci addımdakı təmizləmə hüceyrələrini işə salın, `save_total_limit`-i azaldın |
| Checkpoint-dən model yüklənmir | `config.json` checkpoint qovluğunda yoxdur | 19-cu addımdakı "config.json kopyala" hüceyrəsini işə salın |

Ətraflı konfiqurasiya izahı və layihənin ümumi strukturu üçün **README.md** faylına baxın.